# 📊 Notebook 01: Khảo sát và Phân tích Khám phá Dữ liệu (EDA)
**Đề tài:** Phân tích cảm xúc theo khía cạnh (Aspect-Based Sentiment Analysis - ABSA) trên dữ liệu UIT-ViSFD (phản hồi người dùng về điện thoại).

Mục tiêu của notebook này:
1. Kiểm tra kích thước và tính toàn vẹn của các tập dữ liệu (Train / Dev / Test).
2. Thống kê phân bố tần suất của 10 khía cạnh (`ASPECTS`).
3. Thống kê tỷ lệ phân bố 3 cực tính cảm xúc (`Positive`, `Negative`, `Neutral`) theo từng khía cạnh.
4. Khảo sát 3 khó khăn cốt lõi của đề tài:
   - Phân bố số lượng khía cạnh trên mỗi câu (câu đơn vs đa khía cạnh).
   - Tỷ lệ câu có cảm xúc trái chiều (vừa khen vừa chê).
   - Khảo sát các mẫu rỗng / chỉ chứa `{OTHERS}`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Đảm bảo nạp được module src từ thư mục gốc
sys.path.insert(0, "..")

from src.data.load import load_split
from src.data.schema import ASPECTS, POLARITES

sns.set_theme(style="whitegrid")
plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 1. Nạp dữ liệu và kiểm tra kích thước mẫu

In [ ]:
train_rows = load_split(Path("../data/raw"), "train")
dev_rows = load_split(Path("../data/raw"), "dev")
test_rows = load_split(Path("../data/raw"), "test")

print(f"Tập Train : {len(train_rows):>6} mẫu (Kỳ vọng: 7786)")
print(f"Tập Dev   : {len(dev_rows):>6} mẫu (Kỳ vọng: 1112)")
print(f"Tập Test  : {len(test_rows):>6} mẫu (Kỳ vọng: 2224)")
print(f"Tổng cộng : {len(train_rows) + len(dev_rows) + len(test_rows):>6} mẫu (Kỳ vọng: 11122)")

## 2. Thống kê phân bố độ dài phản hồi (Số từ & Ký tự)

In [ ]:
train_df = pd.DataFrame(train_rows)
train_df["char_len"] = train_df["text"].apply(len)
train_df["word_len"] = train_df["text"].apply(lambda t: len(str(t).split()))

print(train_df[["char_len", "word_len"]].describe())

## 3. Tần suất xuất hiện của 10 khía cạnh trên tập Train

In [ ]:
aspect_counts = {a: 0 for a in ASPECTS}
for r in train_rows:
    for a in r["pairs"]:
        aspect_counts[a] += 1

df_aspects = pd.DataFrame(list(aspect_counts.items()), columns=["Khía cạnh", "Số lượng"])
df_aspects = df_aspects.sort_values(by="Số lượng", ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=df_aspects, x="Khía cạnh", y="Số lượng", palette="viridis")
plt.title("Tần suất xuất hiện của 10 khía cạnh trên tập Train")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Tỷ lệ cực tính cảm xúc (Positive / Negative / Neutral) theo từng khía cạnh

In [ ]:
pol_by_aspect = {a: {p: 0 for p in POLARITES} for a in ASPECTS}
for r in train_rows:
    for a, p in r["pairs"].items():
        pol_by_aspect[a][p] += 1

df_pol = pd.DataFrame(pol_by_aspect).T
df_pol.plot(kind="bar", stacked=True, figsize=(11, 5), color=["#2ecc71", "#e74c3c", "#95a5a6"])
plt.title("Phân bố cảm xúc theo từng khía cạnh trên tập Train")
plt.xlabel("Khía cạnh")
plt.ylabel("Số lượt nhãn")
plt.xticks(rotation=45)
plt.legend(title="Cực tính")
plt.tight_layout()
plt.show()

## 5. Khảo sát 3 bài toán khó đặc thù của đề tài:
- **Đa khía cạnh trong một câu**
- **Cảm xúc trái chiều (vừa khen vừa chê)**
- **Lớp vắng mặt (không có khía cạnh nào)**

In [ ]:
num_aspects_per_doc = [len(r["pairs"]) for r in train_rows]
has_conflict = []
empty_count = 0

for r in train_rows:
    pols = set(r["pairs"].values())
    if len(r["pairs"]) == 0:
        empty_count += 1
    # Trái chiều khi trong cùng 1 câu có cả Positive và Negative
    if "Positive" in pols and "Negative" in pols:
        has_conflict.append(True)
    else:
        has_conflict.append(False)

print(f"• Số mẫu rỗng (không có khía cạnh nào): {empty_count} ({empty_count / len(train_rows):.2%})")
print(f"• Số mẫu chứa cảm xúc trái chiều (+/-): {sum(has_conflict)} ({sum(has_conflict) / len(train_rows):.2%})")
print(f"• Trung bình số khía cạnh trên một câu: {np.mean(num_aspects_per_doc):.2f}")